# Multiclass classification with satellite data 👩🏼‍🚀
The dataset you will be working with comes from the <a href='https://archive.ics.uci.edu/ml/datasets/Statlog+%28Landsat+Satellite%29'>UCI machine learning website</a>. It consists of the multi-spectral values of pixels in 3x3 neighbourhoods in a satellite image, and the classification associated with the central pixel in each neighbourhood. The aim is to predict this classification, given the multi-spectral values. In the sample database, the class of a pixel is coded as a number.

The Landsat satellite data is one of the many sources of information available for a scene. The interpretation of a scene by integrating spatial data of diverse types and resolutions including multispectral and radar data, maps indicating topography, land use etc. is extremely important. Classical statistical methods are ill-equipped for handling such diverse data types.

One frame of Landsat MSS imagery consists of four digital images of the same scene in different spectral bands. Two of these are in the visible region (corresponding approximately to green and red regions of the visible spectrum) and two are in the (near) infra-red. Each pixel is a 8-bit binary word, with 0 corresponding to black and 255 to white. The spatial resolution of a pixel is about 80m x 80m. Each image contains 2340 x 3380 such pixels.

The database is a (tiny) sub-area of a scene, consisting of 82 x 100 pixels. Each line of data corresponds to a 3x3 square neighbourhood of pixels completely contained within the 82x100 sub-area. Each line contains the pixel values in the four spectral bands (converted to ASCII) of each of the 9 pixels in the 3x3 neighbourhood and a number indicating the classification label of the central pixel. The number is a code for the following classes:

Number Class
* 1 red soil
* 2 cotton crop
* 3 grey soil
* 4 damp grey soil
* 5 soil with vegetation stubble
* 6 mixture class (all types present)
* 7 very damp grey soil

NB. There are no examples with class 6 in this dataset. Our local copy uses the descriptive class names instead of the original numerical class codes.

The data is given in random order and certain lines of data have been removed so you cannot reconstruct the original image from this dataset.

In each line of data the four spectral values for the top-left pixel are given first followed by the four spectral values for the top-middle pixel and then those for the top-right pixel, and so on with the pixels read out in sequence left-to-right and top-to-bottom. Thus, the four spectral values for the central pixel are given by attributes 17,18,19 and 20. If you like you can use only these four attributes, while ignoring the others. This avoids the problem which arises when a 3x3 neighbourhood straddles a boundary.


👇 Load the `sat.trn` dataset located within the data folder to see what you will be working with. Note that the dataset again does **NOT** have headers for the columns (you should know how to manage this by now). You will also quickly notice that opening this dataset 'as is' with pandas results in a single column dataframe: this is because the values are not comma separate (cvs) but instead are separated by a space (<code>' '</code>). Read the Pandas doc to see how you can open this file properly.<br><br>
Onced opened, call your new dataframe <code>data</code>. The data has no missing values and is already cleaned for you.

In [ ]:
import pandas as pd
import numpy as np

data = pd.read_csv('../../Excercise_data/L05/sat.trn',sep=r'\s+', header=None)
data

# Visualizing the data
The satellite data that we have consists of 4 channels (4 images) each of 3x3 pixels. So this is a tiny bit of satellite data per each image.<br>
Let's visualise the data for the first image in the dataset. I have already written the code for you. Read the code, make sure you understand it. If not, ask questions. Then run it to see what the data looks like.

In [ ]:
import matplotlib.pyplot as plt

# Four consecutive bands per pixel; pixels run left-to-right, top-to-bottom.
patch = data.iloc[0, :36].to_numpy(dtype=float).reshape(3, 3, 4)
fig, axes = plt.subplots(1, 4, figsize=(10, 3))
for band, ax in enumerate(axes):
    ax.imshow(patch[:, :, band], cmap='gray', vmin=0, vmax=255)
    ax.set_title(f'Band {band + 1}')
    ax.axis('off')
plt.show()

### False-colour images
The four MSS bands are green, red and two near-infrared bands. There is no blue band, so these data cannot produce a natural RGB image.

For a false-colour composite, display band 4 as red, band 2 as green and band 1 as blue. This is an explicit visual mapping, not the original scene's visible colour.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
false_colour = patch[:, :, [3, 1, 0]].astype(np.uint8)
axes[0].imshow(false_colour)
axes[0].set_title('False colour: bands 4, 2, 1')
axes[1].imshow(patch[:, :, 2], cmap='gray', vmin=0, vmax=255)
axes[1].set_title('Band 3: near infrared')
for ax in axes:
    ax.axis('off')
plt.show()

# Preparing the target and splitting
The label column (<code>36</code>) is not encoded. Here, do the following:
* Encode the label column with your own encoder. Save the fitted encoder in a variable named <code>encoder</code> (you will need it later)
* create the features (<code>X</code>) and target (<code>y</code>) dataframes. 
* Normalize <code>X</code> by dividing it by the absolute maximum value of a pixel (read the text above to find out what this is). In cases where there is an absolute maximum value (such as here for pixel values) it is a good idea to normalise to it: this way, all data will always conform to your normalization. As an added bonus, because the minimum value for a pixel is zero, simply dividing by the maximum value results effectively in `MinMax` normalization
* Reserve 30% for testing, then reserve 25% of the remaining development data for validation. Stratify both splits, with random state 42.
* Name the splits `X_train`, `X_val`, `X_test` and their corresponding targets. Use validation for every model comparison; test only the final choice.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

X = data.drop(36, axis=1) / 255
encoder = LabelEncoder().fit(data[36])
y = encoder.transform(data[36])
X_development, X_test, y_development, y_test = train_test_split(
    X, y, random_state=42, test_size=0.3, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(
    X_development, y_development, random_state=42,
    test_size=0.25, stratify=y_development)

# Base Model
Train logistic regression using `OneVsRestClassifier(LogisticRegression(max_iter=5000))`.

OvR fits $K$ binary classifiers; OvO fits $K(K-1)/2$. With six observed classes, these are 6 and 15. Their relative computational cost depends on the learner and the data used by each binary problem. Multinomial logistic regression instead fits a joint multiclass probability model.

Evaluate on **validation** data with accuracy, macro precision, macro recall and macro F1. Use `zero_division=0` where a class receives no predicted labels. Choose **macro F1** as the primary comparison metric before seeing results; report the others to diagnose trade-offs.

Use the explicit `OneVsRestClassifier` wrapper for OvR:

```python
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
ovr_classifier = OneVsRestClassifier(LogisticRegression(max_iter=5000))
ovr_classifier.fit(X_train, y_train)
```

# Confusion Matrix for Multiclass Classification

Analyzing classification errors by looking at raw numbers isn’t always the easiest approach, especially in multiclass classification. Instead, a confusion matrix provides a visual summary of how well your model is performing by showing the number of true and false predictions for each class.

To simplify the process, you will create a function called `make_confusion_matrix` that takes the true labels (`y_true`) and predicted labels (`y_pred`) as inputs and returns a confusion matrix plot.

## Steps:

1. **Use `confusion_matrix` from `sklearn.metrics` to generate the matrix of actual vs predicted values.**
2. **Visualize the matrix using `seaborn.heatmap()` to create a color-coded grid of the confusion matrix.**
3. **Format the matrix to make it clear:**
   - Use the correct labels for the rows and columns (true labels on the y-axis, predicted labels on the x-axis).
   - Add class labels and ensure the numbers inside the matrix (counts of correct/incorrect predictions) are displayed.
   - Use a color map to make the matrix visually appealing (e.g., 'coolwarm').


## *Optional*: improved confusion matrix

Of course, you can vastly improve on the basic design of your confusion matrix. 

Here I added the following:
* Change the size of the confusion matrix using appropriate `matplotlib` objects (`seaborn` is built on top of `matplotlib`)
* In addition to the matrix, I plot the `X` and `Y` axis label, as well as a title
* I added a metric in the title: I chose the average F1 score, as it is a good one to compare different models
* I added a 45$^{\circ}$ tilt to the `x_axis` labels to make the text look nicer, I increased the font of the labels, making some bolds.
* In addition to the total number of samples, I added the percentage of each category (number of sample in each bin divided by total number of samples in a given category (i.e. total of 'true' labels for each category)

You can use your own imagination and sense of graphic design here to make your confusion matrix look good, and be informative. There is no 'canned' method to do this, you need to use some python code and depending on what you want modify the signature of the `make_confusion_matrix` function (in my case, I needed to pass the title of the plot as an additional argument).
<br>
I suggest you do this once you are done with the rest of the notebook: you can then easily redraw your confusion matrix with the newer code.

## Changing the multiclassification strategy
Fit `LogisticRegression(solver='lbfgs', max_iter=5000)` for the joint multinomial model. Compare its validation metrics and confusion matrix with OvR.

In [ ]:
# 

Which approach has the higher **validation macro F1**? Save `ovr` or `multinomial` as `best_approach`. Explain any trade-off with accuracy. Do not average different metrics into an arbitrary composite score.

### Extend your multiclass classification task by evaluating K-Nearest Neighbors (KNN), Support Vector Machine (SVM), and Decision Tree classifiers in addition to Logistic Regression

Choose `final_model` using validation macro F1, then refit it on `X_development, y_development`. Evaluate once on the held-out test set.

Which **class** is most often misclassified, and with which other class? Refer to the confusion matrix. What limitation follows from neighbouring image patches potentially sharing spatial information across a random split?